# nnU-Net BraTS 2024 MEN-RT — Full Pipeline (Kaggle)

**Before running:**
1. GPU must be ON → Notebook Settings → Accelerator → **GPU T4 x2** or **P100**
2. Add your BraTS dataset (the two zip files) as a Kaggle Dataset input
3. Add your GitHub token as a Kaggle Secret named `nnunet_kaggle`

**Pilot plan (50 cases, fold 0 only first):**
- Steps 1–2: prepare + preprocess (~25–45 min)
- Step 3 fold 0: train 50 epochs (~1–2 h on T4)
- Step 4: fold-0 inference → segmentation prompts for foundation model
- Steps 5–6: evaluate + visualize
- Cell 20: optionally train fold 1 with the same 50-epoch budget

**Expected total runtime:** 2–4 hours on a single T4 GPU

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, os, sys

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} | {p.total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU detected. Enable GPU in Notebook Settings before continuing.')

In [ ]:
# ── Cell 2: Auto-discover dataset paths from Kaggle input ─────────────────────
# Scans /kaggle/input/ to find your BraTS zip files automatically.
# No manual editing needed — as long as the dataset is attached.

import glob
from pathlib import Path

def find_kaggle_file(pattern: str) -> str | None:
    """Return the first file matching *pattern* under /kaggle/input."""
    matches = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))
    return matches[0] if matches else None

# ── Find training and validation zips ────────────────────────────────────────
TRAIN_ZIP = find_kaggle_file('BraTS2024-MEN-RT-TrainingData.zip')
VAL_ZIP   = find_kaggle_file('BraTS2024-MEN-RT-ValidationData.zip')

# Fallback: if zips not found, look for extracted directories
if TRAIN_ZIP is None:
    candidates = sorted(glob.glob('/kaggle/input/**/BraTS-MEN-RT-*', recursive=True))
    if candidates:
        # Use the parent of the first case directory as source
        TRAIN_ZIP = str(Path(candidates[0]).parent)

print('Dataset discovery results:')
print(f'  Training source : {TRAIN_ZIP}')
print(f'  Validation zip  : {VAL_ZIP}')

if TRAIN_ZIP is None:
    print()
    print('ERROR: BraTS2024-MEN-RT-TrainingData.zip not found in /kaggle/input/')
    print()
    print('All files in /kaggle/input/:')
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            print(f'  {os.path.join(root, f)}')
    raise FileNotFoundError(
        'Attach your BraTS dataset to this notebook via Add Data → '
        'Your Datasets, then re-run Cell 2.'
    )

print('\nDataset found. Ready to proceed.')

In [ ]:
# ── Cell 3: Clone repository from GitHub ─────────────────────────────────────
# Reads your GitHub token from Kaggle Secrets (secret name: nnunet_kaggle).

from kaggle_secrets import UserSecretsClient

secrets   = UserSecretsClient()
token     = secrets.get_secret('nnunet_kaggle')
repo_url  = f'https://{token}@github.com/maryampervaiz-alt/nnunet-training.git'
REPO_DIR  = '/kaggle/working/nnunet-training'

# Remove stale clone if present, then do a fresh clone
result = subprocess.run(['rm', '-rf', REPO_DIR], capture_output=True)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', repo_url, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(
        f'git clone failed (rc={result.returncode}):\n{result.stderr}\n'
        'Check that your nnunet_kaggle secret contains a valid GitHub token.'
    )

print('Repository cloned successfully.')
print('Contents:')
for item in sorted(Path(REPO_DIR).iterdir()):
    print(f'  {item.name}/'  if item.is_dir() else f'  {item.name}')

In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
# Install order matters:
#   1. surface-distance from GitHub (PyPI package has a different API)
#   2. nnunetv2 pulls torch/numpy/SimpleITK automatically
#   3. Remaining optional packages

import subprocess, sys

def pip_install(pkg: str, quiet: bool = True) -> None:
    args = [sys.executable, '-m', 'pip', 'install', pkg]
    if quiet:
        args.append('-q')
    result = subprocess.run(args, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f'pip install {pkg} failed')

print('Installing surface-distance (from GitHub) ...')
pip_install('git+https://github.com/google-deepmind/surface-distance.git')

print('Installing nnunetv2 ...')
pip_install('nnunetv2')

print('Installing remaining dependencies ...')
extras = [
    'nibabel', 'SimpleITK', 'scipy', 'scikit-image',
    'pandas', 'matplotlib', 'seaborn',
    'loguru', 'rich', 'python-dotenv', 'tqdm',
    'mlflow',
]
pip_install(' '.join(extras))

print('\nAll packages installed.')

In [ ]:
# ── Cell 5: Register custom trainer with nnunetv2 (CRITICAL) ─────────────────
#
# nnU-Net v2 discovers trainer classes by scanning its own package directory
# with recursive_find_python_class(). Our custom nnUNetTrainerEarlyStopping
# lives in src/training/nnunet_trainer_es.py and is NOT inside the nnunetv2
# package, so nnU-Net cannot find it with -tr nnUNetTrainerEarlyStopping.
#
# Fix: copy the trainer file into nnunetv2's training directory.
# This is safe — the file has no relative imports from src/.

import shutil, importlib.util
import nnunetv2
from pathlib import Path

nnunet_pkg_dir    = Path(nnunetv2.__file__).parent
trainer_src       = Path(REPO_DIR) / 'src' / 'training' / 'nnunet_trainer_es.py'
trainer_dst       = nnunet_pkg_dir / 'training' / 'nnUNetTrainer' / 'nnUNetTrainerEarlyStopping.py'

if not trainer_src.exists():
    raise FileNotFoundError(f'Custom trainer source not found: {trainer_src}')

shutil.copy2(trainer_src, trainer_dst)
print(f'Registered: {trainer_src.name} → {trainer_dst}')

# Verify the class is importable
spec = importlib.util.spec_from_file_location('nnUNetTrainerEarlyStopping', trainer_dst)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
cls  = getattr(mod, 'nnUNetTrainerEarlyStopping', None)
if cls is None:
    raise ImportError('nnUNetTrainerEarlyStopping class not found in copied file')

print(f'Trainer class   : {cls}')
print('Custom trainer registration: OK')

In [ ]:
# ── Cell 6: Configure environment variables ───────────────────────────────────

PROJECT  = REPO_DIR
MAX_CASES = 50     # set to None to use ALL training cases

os.environ.update({
    'nnUNet_raw'          : f'{PROJECT}/nnunet_raw',
    'nnUNet_preprocessed' : f'{PROJECT}/nnunet_preprocessed',
    'nnUNet_results'      : f'{PROJECT}/nnunet_results',
    'DATASET_ID'          : '001',
    'DATASET_NAME'        : 'BraTSMENRT',
    'NNUNET_CONFIGURATION': '3d_fullres',
    'NNUNET_SEED'         : '42',
    'ES_PATIENCE'         : '50',
    'ES_MIN_DELTA'        : '0.0001',
    'ES_WARMUP'           : '50',
    'NUM_FOLDS'           : '5',
    'CUDA_VISIBLE_DEVICES': '0',
    'TRAIN_SOURCE'        : str(TRAIN_ZIP),
    'VAL_SOURCE'          : str(VAL_ZIP) if VAL_ZIP else '',
    'LABEL_SUFFIX'        : 'gtv',
    'EXPERIMENT_NAME'     : 'BraTSMENRT_baseline',
    'MLFLOW_TRACKING_URI' : f'{PROJECT}/experiments/mlruns',
})

# Create all required output directories
for d in [
    'nnunet_raw', 'nnunet_preprocessed', 'nnunet_results',
    'metrics', 'results', 'visualizations',
    'inference_outputs', 'logs', 'experiments', 'checkpoints',
]:
    Path(f'{PROJECT}/{d}').mkdir(parents=True, exist_ok=True)

print('Environment configured:')
for k in ['nnUNet_raw', 'nnUNet_preprocessed', 'nnUNet_results',
          'DATASET_ID', 'DATASET_NAME', 'NNUNET_CONFIGURATION']:
    print(f'  {k:25} = {os.environ[k]}')
print(f'  TRAIN_SOURCE              = {TRAIN_ZIP}')
print(f'  MAX_CASES                 = {MAX_CASES}')

In [ ]:
# ── Cell 7: Set working directory to the repo ─────────────────────────────────
# All subsequent ! commands and relative paths resolve from here.

os.chdir(PROJECT)

# Add repo root to Python path so src.* imports work in notebook cells
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print(f'Working directory: {os.getcwd()}')
print(f'Python path includes: {sys.path[0]}')

In [ ]:
# ── Cell 8: STEP 1 — Convert raw BraTS data to nnU-Net format ────────────────
# Converts the first MAX_CASES training cases (alphabetically sorted) plus
# all available validation cases (images only, no labels required).
# Expected time: 2–6 minutes for 50 cases (longer if zips need extracting).

cmd = [
    sys.executable, 'scripts/01_prepare_dataset.py',
    '--train',   str(TRAIN_ZIP),
    '--val',     str(VAL_ZIP) if VAL_ZIP else '',
    '--log-dir', 'logs',
]
if MAX_CASES is not None:
    cmd += ['--max-cases', str(MAX_CASES)]
if not VAL_ZIP:
    cmd.append('--skip-val')

print(f'Running: {" ".join(cmd)}\n')
result = subprocess.run(cmd, env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError(f'01_prepare_dataset.py failed (rc={result.returncode})')
print('\nDataset preparation complete.')

In [ ]:
# ── Cell 9: Verify dataset structure ─────────────────────────────────────────
import json
from pathlib import Path

raw_dir     = Path(os.environ['nnUNet_raw'])
dataset_dir = raw_dir / 'Dataset001_BraTSMENRT'
images_tr   = dataset_dir / 'imagesTr'
labels_tr   = dataset_dir / 'labelsTr'
images_ts   = dataset_dir / 'imagesTs'

n_tr  = len(list(images_tr.glob('*_0000.nii.gz')))
n_lbl = len(list(labels_tr.glob('*.nii.gz')))
n_ts  = len(list(images_ts.glob('*_0000.nii.gz'))) if images_ts.exists() else 0

with open(dataset_dir / 'dataset.json') as fh:
    ds = json.load(fh)

print(f'imagesTr images  : {n_tr}')
print(f'labelsTr labels  : {n_lbl}')
print(f'imagesTs images  : {n_ts}')
print(f'numTraining (json): {ds["numTraining"]}')
print(f'channel_names    : {ds["channel_names"]}')
print(f'labels           : {ds["labels"]}')

assert n_tr == n_lbl, f'Image/label count mismatch: {n_tr} vs {n_lbl}'
assert ds['numTraining'] == n_tr, (
    f'numTraining in dataset.json ({ds["numTraining"]}) does not match '
    f'actual file count ({n_tr})'
)
print('\nDataset structure OK.')

In [ ]:
# ── Cell 10: STEP 2 — nnU-Net Planning & Preprocessing ───────────────────────
# nnU-Net reads the dataset fingerprint and automatically determines:
#   - Target spacing, patch size, batch size, network topology
#   - Normalization scheme (MRI-specific Z-score)
# No manual overrides — let nnU-Net decide everything.
# Expected time: 20–40 minutes for 50 cases.

result = subprocess.run(
    [sys.executable, 'scripts/02_preprocess.py', '--log-dir', 'logs'],
    env=os.environ.copy(),
)
if result.returncode != 0:
    raise RuntimeError(f'02_preprocess.py failed (rc={result.returncode})')
print('Preprocessing complete.')

In [ ]:
# ── Cell 11: Integrity Check — Verify 50-case Subset ─────────────────────────
# Validates every case before training. Checks:
#   • All NIfTI files are readable
#   • Image/label spatial shapes match per case
#   • Label values are in {0, 1} (binary GTV segmentation)
#   • No duplicate case IDs
# If this cell raises an AssertionError, fix the issue BEFORE training.

from src.data.integrity_checker import IntegrityChecker

checker = IntegrityChecker(dataset_dir=dataset_dir, expected_label_values={0, 1})
report  = checker.run()

print('=' * 60)
print('  INTEGRITY CHECK REPORT')
print('=' * 60)
print(report.summary())
print('=' * 60)

failed = [r for r in report.case_reports if not r.ok]
if failed:
    for r in failed:
        print(f'  FAIL [{r.case_id}]: {r.errors}')
    raise AssertionError(
        f'{len(failed)} case(s) failed integrity checks. '
        'Fix the issues above before proceeding.'
    )

print('All cases passed integrity checks. Safe to train.')

In [ ]:
# ── Cell 12: Verify preprocessing + show CV splits ───────────────────────────

preproc_dir = Path(os.environ['nnUNet_preprocessed']) / 'Dataset001_BraTSMENRT'

# List preprocessed files
print('Preprocessed directory contents:')
for item in sorted(preproc_dir.iterdir()):
    print(f'  {item.name}')

# Show splits
splits_file = preproc_dir / 'splits_final.json'
if splits_file.exists():
    with open(splits_file) as fh:
        splits = json.load(fh)
    print(f'\n5-fold cross-validation splits ({len(splits)} folds):')
    for i, fold in enumerate(splits):
        print(f'  Fold {i}: {len(fold["train"])} train | {len(fold["val"])} val')
else:
    print('\nsplits_final.json not found — preprocessing may not have completed.')

# Show nnU-Net plans
for plans_file in sorted(preproc_dir.glob('nnUNetPlans*.json')):
    with open(plans_file) as fh:
        plans = json.load(fh)
    print(f'\nPlans from {plans_file.name}:')
    for cfg_name, cfg in plans.get('configurations', {}).items():
        print(
            f'  [{cfg_name}]  '
            f'patch_size={cfg.get("patch_size")}  '
            f'spacing={cfg.get("spacing")}  '
            f'batch_size={cfg.get("batch_size")}'
        )

In [ ]:
# ── Cell 13: STEP 3 — Train Fold 0 (50 epochs) ───────────────────────────────
#
# Trains fold 0 of the 5-fold CV on the 50-case subset for 50 epochs.
#
# nnU-Net automatically saves:
#   checkpoint_best.pth   — best validation Dice (updated whenever Dice improves)
#   checkpoint_latest.pth — saved every epoch (resume safely after interruption)
#   checkpoint_final.pth  — written at the very end of training
#
# Early stopping: patience=50, warmup=50 epochs
# (with num_epochs=50 the warmup covers the full run, so ES will not trigger)
#
# To RESUME an interrupted run, add '--continue-training' to the command below.
#
# Expected time: ~1–2 hours on T4 GPU (30–40 min on P100)

NUM_EPOCHS = 50
FOLD       = 0

cmd = [
    sys.executable, 'scripts/03_train.py',
    '--folds',          str(FOLD),
    '--num-epochs',     str(NUM_EPOCHS),
    '--seed',           '42',
    '--trainer',        'nnUNetTrainerEarlyStopping',
    '--es-patience',    '50',
    '--es-min-delta',   '0.0001',
    '--es-warmup',      '50',
    '--log-dir',        'logs',
    '--metrics-dir',    'metrics',
    '--checkpoints-dir','checkpoints',
]

print(f'Training fold {FOLD} for {NUM_EPOCHS} epochs ...')
print(f'Command: {" ".join(cmd)}\n')

result = subprocess.run(cmd, env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError(f'Fold {FOLD} training failed (rc={result.returncode})')

print(f'\nFold {FOLD} training complete.')

In [ ]:
# ── Cell 14: Verify fold 0 checkpoints ───────────────────────────────────────
# Checks both the nnU-Net native checkpoint path and the managed archive.

TRAINER   = 'nnUNetTrainerEarlyStopping'
PLANS     = 'nnUNetPlans'
CONFIG    = os.environ['NNUNET_CONFIGURATION']
FOLD      = 0

# ── 1. Native nnU-Net checkpoint directory ────────────────────────────────────
nnunet_ckpt_dir = (
    Path(os.environ['nnUNet_results'])
    / 'Dataset001_BraTSMENRT'
    / f'{TRAINER}__{PLANS}__{CONFIG}'
    / f'fold_{FOLD}'
)

print(f'Native nnU-Net checkpoint dir:')
print(f'  {nnunet_ckpt_dir}')

expected_native = ['checkpoint_best.pth', 'checkpoint_latest.pth']
for fname in expected_native:
    p = nnunet_ckpt_dir / fname
    if p.exists():
        print(f'  OK  {fname:30} ({p.stat().st_size / 1e6:.1f} MB)')
    else:
        print(f'  MISSING  {fname}')

# Show last 15 lines of the training log
log_files = sorted(nnunet_ckpt_dir.glob('training_log*.txt'))
if log_files:
    print(f'\nLast 15 lines of {log_files[-1].name}:')
    with open(log_files[-1]) as fh:
        lines = fh.readlines()
    print(''.join(lines[-15:]))

# ── 2. Managed checkpoint archive (checkpoints/fold_0/) ──────────────────────
print('Managed checkpoint archive:')
managed_dir = Path('checkpoints') / f'fold_{FOLD}'
if managed_dir.exists():
    for f in sorted(managed_dir.iterdir()):
        print(f'  {f.name:30} ({f.stat().st_size / 1e6:.1f} MB)')
else:
    print('  (not yet archived — archive is created by the orchestrator at CV completion)')

print('\nCheckpoint verification complete.')

---
## After Fold 0 Training

With fold 0 trained, proceed to:
- **Cell 15** → Run fold-0 inference → generates `inference_outputs/cv/fold_0/*.nii.gz`
- These predictions are your **segmentation prompts** for the foundation model
- **Cells 16–19** → evaluate metrics, visualize, save outputs
- **Cell 20** → optionally train fold 1 (same 50-epoch budget)

**Resuming after a session interruption:**
Re-run Cells 1–7 (skip 8–12 if data is already preprocessed), then add `'--continue-training'` to the command in Cell 13.

In [ ]:
# ── Cell 15: STEP 4 — Inference using Fold 0 model ───────────────────────────
# Predicts GTV masks on the held-out validation cases of fold 0.
# Output: inference_outputs/cv/fold_0/*.nii.gz
# Expected time: 10–30 minutes

result = subprocess.run(
    [
        sys.executable, 'scripts/04_inference.py',
        '--cv-mode',
        '--folds',    '0',
        '--output',   'inference_outputs/cv',
        '--log-dir',  'logs',
    ],
    env=os.environ.copy(),
)
if result.returncode != 0:
    raise RuntimeError(f'04_inference.py failed (rc={result.returncode})')

pred_dir = Path('inference_outputs/cv/fold_0')
preds    = list(pred_dir.glob('*.nii.gz')) if pred_dir.exists() else []
print(f'Inference complete. {len(preds)} predictions in {pred_dir}')

In [ ]:
# ── Cell 16: STEP 5 — Evaluate Fold 0 ────────────────────────────────────────
# Computes: DSC, HD95, NSD (2 mm), Precision, Recall, Specificity, Volume Error
# Expected time: 5–15 minutes

result = subprocess.run(
    [
        sys.executable, 'scripts/05_evaluate.py',
        '--cv-mode',
        '--results-dir',  'results',
        '--show-rankings',
        '--log-dir',      'logs',
    ],
    env=os.environ.copy(),
)
if result.returncode != 0:
    raise RuntimeError(f'05_evaluate.py failed (rc={result.returncode})')

print('Evaluation complete.')

In [ ]:
# ── Cell 17: STEP 6 — Visualizations ─────────────────────────────────────────
# Generates: segmentation overlays, violin/box plots, training curves
# Expected time: 2–5 minutes

result = subprocess.run(
    [
        sys.executable, 'scripts/06_visualize.py',
        '--all',
        '--cv-mode',
        '--results-dir',  'results',
        '--metrics-dir',  'metrics',
        '--output-dir',   'visualizations',
        '--log-dir',      'logs',
    ],
    env=os.environ.copy(),
)
if result.returncode != 0:
    print(f'Warning: 06_visualize.py exited with rc={result.returncode} '
          '(non-fatal — some plots may be missing)')

In [ ]:
# ── Cell 18: View quantitative results ───────────────────────────────────────

import pandas as pd

for csv_file in ['results/fold_0_per_case.csv', 'results/cv_combined.csv']:
    p = Path(csv_file)
    if not p.exists():
        print(f'{csv_file}: not found')
        continue
    df = pd.read_csv(p)
    cols = [c for c in ['case_id', 'dice', 'hd95', 'nsd', 'precision', 'recall'] if c in df.columns]
    print(f'\n=== {csv_file} ({len(df)} cases) ===')
    print(df[cols].round(4).to_string(index=False))
    print()
    if 'dice' in df.columns:
        print(f'  Mean DSC  : {df["dice"].mean():.4f} ± {df["dice"].std():.4f}')
    if 'hd95' in df.columns:
        finite_hd = df['hd95'].replace([float('inf')], float('nan'))
        print(f'  Mean HD95 : {finite_hd.mean():.2f} mm')
    if 'nsd' in df.columns:
        print(f'  Mean NSD  : {df["nsd"].mean():.4f}')

In [ ]:
# ── Cell 19: Show overlay images and metric plots ─────────────────────────────

from IPython.display import Image, display

# Show up to 3 segmentation overlays
overlay_dir = Path('visualizations/overlays')
if overlay_dir.exists():
    for img_path in sorted(overlay_dir.glob('*.png'))[:3]:
        print(f'\n{img_path.name}')
        display(Image(str(img_path), width=900))
else:
    print('No overlay images found — run Cell 17 first')

# Show metric plots
for plot in [
    'visualizations/metrics_violin.png',
    'visualizations/metrics_boxplot.png',
    'visualizations/volume_scatter.png',
    'visualizations/training_curves.png',
]:
    if Path(plot).exists():
        print(f'\n{plot}')
        display(Image(plot, width=900))

In [ ]:
# ── Cell 20: Save all outputs to /kaggle/working ──────────────────────────────
# Kaggle deletes /kaggle/working/nnunet-training when the session ends.
# This cell copies everything important to /kaggle/working/outputs which IS
# preserved as a notebook output and can be downloaded.

import shutil

SAVE = '/kaggle/working/outputs'

to_save = [
    ('results',             f'{SAVE}/results'),
    ('metrics',             f'{SAVE}/metrics'),
    ('visualizations',      f'{SAVE}/visualizations'),
    ('inference_outputs',   f'{SAVE}/inference_outputs'),
    ('logs',                f'{SAVE}/logs'),
    ('checkpoints',         f'{SAVE}/checkpoints'),
    # Also save the native nnU-Net checkpoints (checkpoint_best.pth, etc.)
    (os.environ['nnUNet_results'], f'{SAVE}/nnunet_results'),
]

Path(SAVE).mkdir(parents=True, exist_ok=True)

for src, dst in to_save:
    if Path(src).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        # Count files saved
        n = sum(1 for _ in Path(dst).rglob('*') if Path(_).is_file())
        print(f'Saved {n:4d} files: {src} → {dst}')
    else:
        print(f'Skipped (not found): {src}')

print(f'\nAll outputs saved to {SAVE}')

# Summary of checkpoint files
print('\nCheckpoint files:')
for p in sorted(Path(SAVE).rglob('*.pth')):
    print(f'  {p.relative_to(SAVE)} ({p.stat().st_size / 1e6:.1f} MB)')

---
## Train Fold 1 (Cell 21)

Run Cell 21 **after** fold-0 training + inference + evaluation are complete.

**In a new Kaggle session:** re-run Cells 1–7, then run Cell 21 directly
(skip Cells 8–12 — preprocessing output persists from previous session as long
as you saved it to `/kaggle/working/outputs/`).

> **Resume tip:** if fold 1 training is interrupted, add `'--continue-training'` to the command.

In [ ]:
# ── Cell 21: Train Fold 1 (50 epochs) ────────────────────────────────────────

NUM_EPOCHS = 50
FOLD       = 1

cmd = [
    sys.executable, 'scripts/03_train.py',
    '--folds',           str(FOLD),
    '--num-epochs',      str(NUM_EPOCHS),
    '--seed',            '42',
    '--trainer',         'nnUNetTrainerEarlyStopping',
    '--es-patience',     '50',
    '--es-min-delta',    '0.0001',
    '--es-warmup',       '50',
    '--log-dir',         'logs',
    '--metrics-dir',     'metrics',
    '--checkpoints-dir', 'checkpoints',
]

print(f'Training fold {FOLD} for {NUM_EPOCHS} epochs ...')
print(f'Command: {" ".join(cmd)}\n')

result = subprocess.run(cmd, env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError(f'Fold {FOLD} training failed (rc={result.returncode})')

print(f'\nFold {FOLD} training complete.')
print(f'  Checkpoints  : checkpoints/fold_{FOLD}/')
print(f'  Training log : metrics/fold_{FOLD}_training.csv')
print('\nNext: run Cell 15 (inference) and Cell 16 (evaluation) for this fold.')